# 1. Dependencies

## CUDA

In [1]:
use_cuda = True

In [2]:
import os
import sys

In [3]:
if use_cuda:
    _cuda_root = os.path.join(sys.prefix, "targets", "x86_64-linux")
    if os.path.isdir(os.path.join(_cuda_root, "include")):
        os.environ.setdefault("CUDA_PATH", _cuda_root)

    %load_ext cuml.accel

cuML: Accelerator installed.


## Common Libraries

In [4]:
import gc
import glob
import time
from contextlib import contextmanager
from typing import cast, Any

In [5]:
import json
import joblib
from joblib import Parallel, delayed, parallel_config
from joblib import parallel_config

In [6]:
import math
import numpy as np
import pandas as pd
import polars as pl
import polars.selectors as cs

## Plotting

In [7]:
import matplotlib.pyplot as plt

## Pre-Processing

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import IncrementalPCA

# from imblearn.over_sampling import SMOTE

## Models

### KNN

In [9]:
if use_cuda:
    from cuml.neighbors import KNeighborsClassifier, NearestNeighbors
else:
    from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors

### SVM

In [10]:
# cuml.accel does wrap sklearn's SVC, but its proxy refuses multiclass outright:
# `_gpu_fit` raises UnsupportedOnGPU as soon as `len(np.unique(y)) > 2` and the call
# silently falls back to CPU libsvm. With 15 attack classes that fallback fires on
# every single fit, so the accelerated import never once reaches the GPU.
#
# cuml.svm.SVC does support multiclass: `fit` routes n_classes > 2 through
# cuml.multiclass.MulticlassClassifier, and each of the resulting pairwise binary
# SVMs is fitted on the GPU. `decision_function_shape` picks the strategy and
# defaults to "ovo", the same one-vs-one scheme libsvm uses, so the model this
# builds is comparable with the CPU one.
if use_cuda:
    from cuml.svm import SVC
else:
    from sklearn.svm import SVC

## Evaluation

In [11]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
)

## Global Variables

### Path

In [12]:
PATH_CACHE = "cache"

In [13]:
PATH_FOLDER_CSV = "cse-cic-ids2018"
PATH_FOLDER_RAW = "data-raw"

In [14]:
PATH_FOLDER_NO_INF = "data-no-inf"

In [15]:
PATH_FOLDER_SPLIT_DATA = "data-split-feature"
PATH_FOLDER_SPLIT_LABEL = "data-split-label"

In [16]:
PATH_FOLDER_ENCODED_LABEL = "data-encoded-label"
PATH_LABEL_ENCODER = os.path.join(PATH_CACHE,"label-encoder.pkl")

In [17]:
PATH_IMPUTER = os.path.join(PATH_CACHE,"median-imputer.json")
PATH_FOLDER_IMPUTED = "data-imputed"

In [18]:
PATH_SCALER = os.path.join(PATH_CACHE,"standard-scaler.pkl")
PATH_FOLDER_SCALED = "data-scaled"

In [19]:
PATH_FOLDER_IPCA_TRANSFORMER = os.path.join(PATH_CACHE,"pca")
PATH_IPCA = "ipca-transformer.pkl"
PATH_FOLDER_IPCA = "data-ipca"

In [20]:
PATH_FOLDER_SMOTE_TRANSFORMER = os.path.join(PATH_CACHE, "smote")
PATH_SMOTE = "smote-transformer.pkl"

In [21]:
PATH_FOLDER_SMOTE = "data-smote"

In [22]:
PATH_FOLDER_MODEL = "trained-model"

In [23]:
PATH_FOLDER_PREDICTION_RESULT = "prediction-result"

In [24]:
PATH_FOLDER_SEARCH_RESULT = "hyperparameter-search"

### Others

In [25]:
LABEL_COLUMN = "Label"
PARQUET_FLOAT_DTYPE = "float32"

## Helper Functions

In [71]:
def get_all_file_names(source_folder_name: str, file_format: str = ""):
    """List file names in source_folder_name that end with `.{file_format}`, sorted alphabetically.

    Note: if file_format is left empty, this returns an empty list (files are only ever
    appended when a format is given), so a format should always be supplied.
    """
    files = []
    for file in os.listdir(source_folder_name):
        if len(file_format) > 0:
            if file.endswith(f".{file_format}"):
                files.append(file)
    return sorted(files)

In [72]:
def filter_file_names(list_file_names: list, filter_word: str):
    """Return only the file names that contain filter_word as a substring (e.g. a date like "2018-02-14")."""
    filtered_file_names = []
    for file_name in list_file_names:
        if filter_word in file_name:
            filtered_file_names.append(file_name)
    return filtered_file_names

In [73]:
def to_pandas_dataframe(data, template_column: list) -> pd.DataFrame:
    """Build a DataFrame from `data` and reindex its columns to match `template_column`."""
    df = pd.DataFrame(data)
    df = df.reindex(columns=template_column)
    return df

In [74]:
def get_split_parquet_files(source_folder_name: str, split_name: str) -> list[str]:
    """Return sorted paths to all Parquet files under `source_folder_name/split_name` (e.g. .../train)."""
    files = sorted(glob.glob(os.path.join(source_folder_name, split_name, "*.parquet")))
    if not files:
        raise FileNotFoundError(
            f"No Parquet files found in {os.path.join(source_folder_name, split_name)}"
        )
    print(f"Found {len(files)} {split_name} files")
    return files

In [75]:
def get_polars_data_frame_with_label(
    source_folder_name: str,
    split_name: str = "train",
    label_column: str = LABEL_COLUMN,
    downcast_to_float32: bool = True,
) -> tuple[pl.DataFrame, pl.Series]:

    label_column = label_column.lower()
    files = get_split_parquet_files(source_folder_name, split_name)

    lazy_frame = pl.concat(pl.scan_parquet(file) for file in files)
    if downcast_to_float32:
        lazy_frame = lazy_frame.with_columns(cs.numeric().cast(pl.Float32))

    dataframe = lazy_frame.collect(engine="streaming")
    if label_column not in dataframe.columns:
        raise ValueError(f"Label column '{label_column}' not found in dataframe.")
    x = dataframe.drop(label_column)
    y = dataframe[label_column]
    return x, y

In [76]:
def get_polars_data_frame_without_label(
    source_folder_name: str, split_name: str = "", downcast_to_float32: bool = True
) -> pl.DataFrame:
    if len(split_name) > 0:
        files = get_split_parquet_files(source_folder_name, split_name)
    else:
        files = get_all_file_names(source_folder_name)
    total = len(files)

    lazy_frames = []
    for i, file in enumerate(files, start=1):
        print(f"Processing [{i}/{total}] {file}...", end="\r", flush=True)
        lazy_frames.append(pl.scan_parquet(file))
    lazy_frame = pl.concat(lazy_frames, how="diagonal_relaxed")

    if downcast_to_float32:
        lazy_frame = lazy_frame.with_columns(cs.numeric().cast(pl.Float32))

    dataframe = lazy_frame.collect(engine="streaming")

    return dataframe

### Model Training

In [77]:
def check_openmp_threads() -> int:
    """Report the thread count that actually governs the KNN search.

    Once sklearn dispatches a brute-force search to `ArgKmin.compute()` it
    threads with OpenMP and ignores `n_jobs` entirely. If this prints 1 the
    search runs single-threaded and will take roughly `n_cores` times longer.
    """
    from sklearn.utils._openmp_helpers import _openmp_effective_n_threads

    n_threads = _openmp_effective_n_threads()
    print(f"OpenMP effective threads: {n_threads}")
    try:
        import threadpoolctl

        for info in threadpoolctl.threadpool_info():
            print(
                f"  {info['user_api']:>8} / {info['internal_api']:<12}"
                f" threads={info['num_threads']}"
            )
    except ImportError:
        print("  (pip install threadpoolctl for per-library detail)")
    return n_threads

In [78]:
def dump_trained_model(
    model, name: str, subfolder: str, folder: str = PATH_FOLDER_MODEL
):
    target_folder = os.path.join(folder, subfolder)
    os.makedirs(target_folder, exist_ok=True)
    file_path = os.path.join(target_folder, name)
    joblib.dump(model, file_path)
    return file_path

In [79]:
def free_gpu_memory() -> None:
    """Release the device memory a finished cuML model was holding.

    cuML allocates through RMM/CuPy pools that only hand memory back once the Python
    objects are actually collected, so `del model` alone is not enough: measured over
    the 9-combination grid the card grew by roughly 850 MiB per fit
    (970 -> 1,826 -> 2,682 -> ... -> 6,424 MiB) and never came down. Collecting and
    then draining the pools returns it to ~106 MiB after every combination.

    This is about staying inside the 8 GiB card on longer grids and bigger
    subsamples. It is *not* the fix for the "Working set has already been
    initialized!" failure - that one was reproduced with the card at 166 MiB, so it
    is a solver bug, not exhaustion. The search guards against it separately.
    """
    gc.collect()
    if not use_cuda:
        return
    try:
        import cupy as cp

        cp.get_default_memory_pool().free_all_blocks()
        cp.get_default_pinned_memory_pool().free_all_blocks()
    except Exception:
        # Freeing is an optimisation, never a reason to kill a long search.
        pass

In [80]:
@contextmanager
def step(message: str):
    """Announce a blocking stage, then tick it off when it returns.

    `SVC.fit` is one opaque call, so there is nothing inside it to count. Printing
    `message ...` *before* entering it and completing the line afterwards is enough to
    tell which stage is running and which stages are finished - no timer and no
    background thread, just the two things that are actually known.
    """
    print(f"  {message} ...", end="", flush=True)
    try:
        yield
    except BaseException:
        print(" failed")
        raise
    print(" done")


def progress_bar(done: int, total: int, label: str = "", width: int = 30) -> None:
    """Redraw a one-line bar. Driven by the caller's own loop, so no thread is needed.

    Ends the line on the final update so whatever prints next starts cleanly.
    """
    filled = width if total <= 0 else int(width * done / total)
    line = f"  [{'#' * filled}{'-' * (width - filled)}] {done:,}/{total:,}"
    if label:
        line += f" {label}"
    print(line.ljust(100), end="\n" if done >= total else "\r", flush=True)

### Model Evaluation

In [81]:
def _unique_int_labels(true, pred) -> list[int]:
    return sorted({int(x) for x in np.unique(true)} | {int(x) for x in np.unique(pred)})

In [82]:
def evaluate(pred, true, labels=None) -> pd.DataFrame:
    if labels is None:
        labels = _unique_int_labels(true, pred)
    return pd.DataFrame(
        [
            {
                "accuracy": accuracy_score(true, pred),
                "precision_macro": precision_score(
                    true, pred, labels=labels, average="macro", zero_division=0
                ),
                "recall_macro": recall_score(
                    true, pred, labels=labels, average="macro", zero_division=0
                ),
                "f1_macro": f1_score(
                    true, pred, labels=labels, average="macro", zero_division=0
                ),
            }
        ]
    )

In [83]:
def get_classification_report(
    pred, true, label_encoder_path: str = PATH_LABEL_ENCODER
) -> pd.DataFrame:
    labels = _unique_int_labels(true, pred)
    precision, recall, f1, support = precision_recall_fscore_support(
        true, pred, labels=labels, average=None, zero_division=0
    )
    return pd.DataFrame(
        {
            "class": decode_labels(labels, label_encoder_path),
            "precision": precision,
            "recall": recall,
            "f1": f1,
            "support": support,
        }
    )

In [84]:
def get_confusion_matrix(
    pred, true, label_encoder_path: str = PATH_LABEL_ENCODER
) -> pd.DataFrame:
    labels = _unique_int_labels(true, pred)
    class_names = decode_labels(labels, label_encoder_path)
    cm = confusion_matrix(true, pred, labels=labels)
    return pd.DataFrame(cm, index=class_names, columns=class_names)

# 2. Pre-Processing

## 2.1. Change CSV to Parquet

In [ ]:
CHUNK_SIZE = 100000
COLUMNS_TO_DROP = {
    "flow id",
    "src ip",
    "source ip",
    "src port",
    "source port",
    "dst ip",
    "destination ip",
    "timestamp",
}

In [ ]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df.columns = df.columns.str.strip().str.lower()
    return df

In [ ]:
def drop_unwanted_columns(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(columns=COLUMNS_TO_DROP, errors="ignore")

In [ ]:
def dataframe_to_parquet(
    chunk: pd.DataFrame,
    target_folder_name: str,
    file_name: str,
    template_columns: list[str] | None = None,
):
    chunk = normalize_columns(chunk)
    chunk = drop_unwanted_columns(chunk)

    if template_columns is not None:
        chunk = chunk.reindex(columns=template_columns)

    label = LABEL_COLUMN.lower()
    for col in chunk.columns:
        if col != label:
            chunk[col] = pd.to_numeric(chunk[col], errors="coerce").astype("float64")

    chunk = chunk[chunk[label].str.lower() != label]
    chunk = chunk.dropna(subset=[label])
    output_file = os.path.join(target_folder_name, file_name)
    chunk.to_parquet(output_file, engine="pyarrow", compression="snappy", index=False)

In [ ]:
def create_template_columns(files: list, source_folder_name: str):
    template_columns = []
    seen = set()
    for file_name in files:
        source_file = os.path.join(source_folder_name, file_name)
        cols = (
            pd.read_csv(source_file, nrows=0).columns.str.strip().str.lower().tolist()
        )
        for c in cols:
            if c not in seen and c not in COLUMNS_TO_DROP:
                seen.add(c)
                template_columns.append(c)
    return template_columns

In [ ]:
def get_template_columns(
    source_folder_name, template_column_path: str = "dataframe-index.pkl"
):
    csv_files = get_all_file_names(source_folder_name, "csv")
    template_columns = create_template_columns(csv_files, source_folder_name)
    joblib.dump(template_columns, template_column_path)

In [ ]:
def _convert_single_csv_file(
    source_folder_name: str,
    target_folder_name: str,
    file_name: str,
    template_columns: list[str],
    chunk_size: int,
):
    source_file = os.path.join(source_folder_name, file_name)
    base_name = os.path.splitext(file_name)[0]

    for chunk_number, chunk in enumerate(
        pd.read_csv(source_file, chunksize=chunk_size, low_memory=False), start=1
    ):
        dataframe_to_parquet(
            chunk,
            target_folder_name,
            f"{base_name}_{chunk_number:05d}.parquet",
            template_columns,
        )

In [ ]:
def convert_all_file_to_parquet(
    source_folder_name: str, target_folder_name: str, chunk_size: int = CHUNK_SIZE
):
    csv_files = get_all_file_names(source_folder_name, "csv")
    template_columns = create_template_columns(csv_files, source_folder_name)

    total = len(csv_files)
    os.makedirs(target_folder_name, exist_ok=True)

    with parallel_config(backend="loky", inner_max_num_threads=1, verbose=0):
        Parallel(n_jobs=-1, verbose=5)(
            delayed(_convert_single_csv_file)(
                source_folder_name,
                target_folder_name,
                file_name,
                template_columns,
                chunk_size,
            )
            for file_name in csv_files
        )

    print(f"Finished processing {total} files.")

In [ ]:
convert_all_file_to_parquet(PATH_FOLDER_CSV, PATH_FOLDER_RAW, CHUNK_SIZE)

## 2.2 Exploratory Data Analysis (EDA)

## 2.3. Change Infinite Values to NaN


Infinite values ($\infty$ and $-\infty$ / `np.inf` and `-np.inf`) are converted to `NaN` values to ensure that they can be handled consistently during the subsequent missing-value imputation process. This transformation allows both originally missing values and invalid infinite values to be processed using the same imputation method.

In [ ]:
def calculate_inf_values(source_folder_name: str):
    parquet_files = get_all_file_names(source_folder_name, "parquet")
    total = len(parquet_files)
    count = 0
    for i, file_name in enumerate(parquet_files, start=1):
        print(f"Processing [{i}/{total}] {file_name}...", end="\r", flush=True)
        source_file = os.path.join(source_folder_name, file_name)
        df = pd.read_parquet(source_file)
        inf_count = np.isinf(df.select_dtypes(include=np.number)).sum().sum()
        count += inf_count
    print("\n")
    print(f"Completed. Processed {total} files.")
    print(f"Found {count} inf values")

In [ ]:
def _change_inf_to_nan_single_file(
    source_folder_name: str, target_folder_name: str, file_name: str
):
    source_file = os.path.join(source_folder_name, file_name)
    target_file = os.path.join(target_folder_name, file_name)

    df = pd.read_parquet(source_file)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.to_parquet(target_file, index=False)


def change_inf_to_nan(source_folder_name: str, target_folder_name: str):
    os.makedirs(target_folder_name, exist_ok=True)

    parquet_files = get_all_file_names(source_folder_name, "parquet")
    total = len(parquet_files)

    with parallel_config(backend="loky", inner_max_num_threads=1, verbose=0):
        Parallel(n_jobs=-1, verbose=5)(
            delayed(_change_inf_to_nan_single_file)(
                source_folder_name, target_folder_name, file_name
            )
            for file_name in parquet_files
        )

    print(f"Completed. Processed {total} files.")

In [ ]:
change_inf_to_nan(PATH_FOLDER_RAW, PATH_FOLDER_NO_INF)

### Evaluation

The dataset was cleaned by converting both positive infinite  and negative infinite values ($\infty$ and $-\infty$ / `np.inf` and `-np.inf`) to `NaN`. A total of **121,886 infinite values** were identified across **168 Parquet files** before the cleaning process. After the transformation, no infinite values remained in the dataset.

**Before cleaning:**

In [ ]:
calculate_inf_values(PATH_FOLDER_RAW)

**After cleaning:**

In [ ]:
calculate_inf_values(PATH_FOLDER_NO_INF)

This ensures that all infinite values are handled as missing values and can subsequently be processed during the missing-value imputation stage.


## 2.4. Train/Validation/Test Split

The CSE-CIC-IDS2018 dataset requires careful consideration when dividing the data into training, validation, and test sets. This is because the attack classes are not uniformly distributed across the dataset; instead, specific attack scenarios were conducted on particular dates. As a result, directly splitting the dataset based on individual days may cause some attack classes to be absent from one or more subsets.

The distribution of attack scenarios across the data collection dates is presented below:

| Date  | Attack(s)                           |
|-------|-------------------------------------|
| 14-02 | FTP-BruteForce, SSH-Bruteforce      |
| 15-02 | DoS-GoldenEye, DoS-Slowloris        |
| 16-02 | DoS-SlowHTTPTest, DoS-Hulk          |
| 20-02 | DDoS-LOIC-HTTP, DDoS-LOIC-UDP       |
| 21-02 | DDoS-LOIC-UDP, DDoS-HOIC            |
| 22-02 | Web Brute Force, XSS, SQL Injection |
| 23-02 | Web Brute Force, XSS, SQL Injection |
| 28-02 | Infiltration                        |
| 01-03 | Infiltration                        |
| 02-03 | Bot                                 |

This distribution indicates that several attack classes are associated with only one or a small number of collection dates. Therefore, assigning entire dates directly to the training, validation, or test set could result in certain attack classes being completely absent from the training data. Such a split would make the experiment evaluate unseen attack-class generalization rather than the intended robustness of the ML-IDS against input disturbances.

Therefore, the primary experiment uses a **stratified train/validation/test split based on the attack label**, ensuring that the attack classes are represented across the three subsets. The validation and test sets are kept separate from the training data to prevent information leakage during model development and final evaluation.

A separate day- or scenario-based split may subsequently be used as an additional experiment to evaluate the model's ability to generalize to traffic collected under different attack scenarios.


In [ ]:
bruteforce = "2018-02-14"
dos_golden = "2018-02-15"
dos_hulk = "2018-02-16"
ddos_http = "2018-02-20"
ddos_udp = "2018-02-21"
web_first = "2018-02-22"
web_second = "2018-02-23"
infiltration_first = "2018-02-28"
infiltration_second = "2018-03-01"
botnet = "2018-03-02"

In [ ]:
def create_train_dev_test_folder(
    source_folder_name: str,
    features_folder_name: str,
    labels_folder_name: str,
    target_day,
):

    parquet_files = get_all_file_names(source_folder_name, "parquet")
    parquet_files = filter_file_names(parquet_files, target_day)

    if not parquet_files:
        print(f"No Parquet files found for {target_day}")
        raise ValueError(f"No Parquet files found for {target_day}")
    print(f"Found {len(parquet_files)} files for {target_day}")

    split_names = ("train", "dev", "test")
    feature_folders = {}
    label_folders = {}
    for split_name in split_names:
        feature_folder = os.path.join(features_folder_name, split_name)
        label_folder = os.path.join(labels_folder_name, split_name)
        os.makedirs(feature_folder, exist_ok=True)
        os.makedirs(label_folder, exist_ok=True)
        feature_folders[split_name] = feature_folder
        label_folders[split_name] = label_folder

    return parquet_files, feature_folders, label_folders

In [ ]:
def _split_single_parquet_file(
    source_folder_name: str,
    file_name: str,
    feature_folders: dict[str, str],
    label_folders: dict[str, str],
    label_column: str,
    dev_size: float,
    test_size: float,
    random_state: int,
):
    source_file = os.path.join(source_folder_name, file_name)
    df = pd.read_parquet(source_file)

    train_df, temp_df = train_test_split(
        df,
        test_size=dev_size + test_size,
        random_state=random_state,
        stratify=df[label_column],
    )
    dev_df, test_df = train_test_split(
        temp_df,
        test_size=test_size / (dev_size + test_size),
        random_state=random_state,
        stratify=temp_df[label_column],
    )

    for split_name, split_df in (
        ("train", train_df),
        ("dev", dev_df),
        ("test", test_df),
    ):
        labels = split_df[[label_column]]
        features = split_df.drop(columns=[label_column])

        features.to_parquet(
            os.path.join(feature_folders[split_name], file_name), index=False
        )
        labels.to_parquet(
            os.path.join(label_folders[split_name], file_name), index=False
        )

In [ ]:
def split_parquet_files(
    source_folder_name: str,
    parquet_files: list[str],
    feature_folders: dict[str, str],
    label_folders: dict[str, str],
    label_column: str,
    dev_size: float,
    test_size: float,
    random_state: int,
):
    total = len(parquet_files)

    with parallel_config(backend="loky", inner_max_num_threads=1, verbose=0):
        Parallel(n_jobs=-1, verbose=5)(
            delayed(_split_single_parquet_file)(
                source_folder_name,
                file_name,
                feature_folders,
                label_folders,
                label_column,
                dev_size,
                test_size,
                random_state,
            )
            for file_name in parquet_files
        )

    print(f"Completed splitting {total} files.")

In [ ]:
def split_a_single_day(
    source_folder_name: str,
    features_folder_name: str,
    labels_folder_name: str,
    target_day: str,
    train_size: float = 0.60,
    dev_size: float = 0.20,
    test_size: float = 0.20,
    label_column: str = LABEL_COLUMN.lower(),
    random_state: int = 42,
):
    if min(train_size, dev_size, test_size) < 0.0:
        raise ValueError("train_size, dev_size, test_size must be non-negative")
    if not math.isclose(train_size + dev_size + test_size, 1.0, abs_tol=1e-9):
        raise ValueError("train_size + dev_size + test_size must equal 1.0")

    parquet_files, feature_folders, label_folders = create_train_dev_test_folder(
        source_folder_name, features_folder_name, labels_folder_name, target_day
    )

    split_parquet_files(
        source_folder_name,
        parquet_files,
        feature_folders,
        label_folders,
        label_column,
        dev_size,
        test_size,
        random_state,
    )

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, bruteforce)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, dos_golden)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, dos_hulk)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, ddos_http)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, ddos_udp)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, web_first)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, web_second)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, infiltration_first)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, infiltration_second)

In [ ]:
split_a_single_day(PATH_FOLDER_NO_INF, PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_SPLIT_LABEL, botnet)

Files are successfully split.

## 2.5. Imputation

Missing values are filled using **per-column medians computed from the training split only**, then applied to train/dev/test. Using the median (rather than the mean) avoids distortion from the heavy outliers typical of network flow features (e.g. flow duration, byte counts). Fitting the medians on the training split only and reusing them for dev/test prevents information leakage from the validation/test sets into preprocessing.

### Create Imputation

imputation for NaN

In [ ]:
def get_numeric_columns(combined: pl.DataFrame) -> list[str]:
    schema = combined.collect_schema()

    numeric_cols = []
    for column, dtype in zip(schema.names(), schema.dtypes()):
        if dtype.is_numeric():
            numeric_cols.append(column)

    print(f"Found {len(numeric_cols)} numeric columns")
    return numeric_cols

In [ ]:
def compute_medians(combined: pl.DataFrame, numeric_cols: list[str]) -> pl.DataFrame:
    median_exprs = []
    total = len(numeric_cols)

    for i, column in enumerate(numeric_cols, start=1):
        print(f"Processing [{i}/{total}] {column}...", end="\r", flush=True)
        expr = pl.col(column).median().alias(column)
        median_exprs.append(expr)

    medians = combined.select(median_exprs)
    return medians

In [ ]:
def build_median_dict(
    medians: pl.DataFrame, numeric_cols: list[str]
) -> dict[str, float]:
    median_dict = {}
    null_columns = []

    for c in numeric_cols:
        val = medians[c][0]
        if val is None:
            null_columns.append(c)
            median_dict[c] = 0.0
        else:
            median_dict[c] = float(val)

    if null_columns:
        print(
            f"Warning: {len(null_columns)} columns had no non-null values, defaulted to 0.0: {null_columns}"
        )

    return median_dict

In [ ]:
def save_median_dict(median_dict: dict[str, float], output_file: str):
    with open(output_file, "w") as file:
        json.dump(median_dict, file, indent=4)
    print(f"Saved medians to: {output_file}")

In [ ]:
def get_median_imputation(
    source_folder_name: str, output_file: str
) -> dict[str, float]:
    df = get_polars_data_frame_without_label(source_folder_name, "train")
    numeric_cols = get_numeric_columns(df)
    medians = compute_medians(df, numeric_cols)
    median_dict = build_median_dict(medians, numeric_cols)
    save_median_dict(median_dict, output_file)
    return median_dict

In [ ]:
median = get_median_imputation(PATH_FOLDER_SPLIT_DATA, PATH_IMPUTER)
print(json.dumps(median, indent=4))

### Impute Training Data

In [ ]:
def impute_dataframe(df: pd.DataFrame, medians):
    for column, median in medians.items():
        if column in df.columns:
            df[column] = df[column].fillna(median)
    return df

In [ ]:
def _impute_single_file(
    source_file: str, output_split_folder: str, medians: dict[str, float]
):
    df = pd.read_parquet(source_file)
    df = impute_dataframe(df, medians)
    output_file = os.path.join(output_split_folder, os.path.basename(source_file))
    df.to_parquet(output_file, engine="pyarrow", compression="snappy", index=False)


def impute_all_split_files(
    source_folder_name: str, output_folder_name: str, medians_path: str
):
    with open(medians_path, "r") as f:
        medians = json.load(f)

    for split_name in ("train", "dev", "test"):
        files = get_split_parquet_files(source_folder_name, split_name)
        total = len(files)
        output_split_folder = os.path.join(output_folder_name, split_name)
        os.makedirs(output_split_folder, exist_ok=True)

        with parallel_config(backend="loky", inner_max_num_threads=1, verbose=0):
            Parallel(n_jobs=-1, verbose=5)(
                delayed(_impute_single_file)(file, output_split_folder, medians)
                for file in files
            )

        print(f"Finished imputing {total} {split_name} files.")

In [ ]:
impute_all_split_files(PATH_FOLDER_SPLIT_DATA, PATH_FOLDER_IMPUTED, PATH_IMPUTER)

## 2.6. Label Encoding

### Create Label Encoder

The string attack labels written by the train/dev/test split (`PATH_FOLDER_SPLIT_LABEL`) are mapped to integer class ids with `LabelEncoder`. As with every other transformer in this pipeline the encoder is fit on the **train split only** and then reused for dev/test, so the class-id mapping never depends on data the model is evaluated on. The fitted encoder is pickled to `PATH_LABEL_ENCODER` so predictions can later be turned back into human-readable labels with `decode_labels` / `encoder.inverse_transform`.

In [ ]:
def create_label_encoder(
    labels_folder_name: str, split_name: str = "train"
) -> LabelEncoder:
    labels = get_polars_data_frame_without_label(
        labels_folder_name, split_name
    ).to_series()

    encoder = LabelEncoder()
    encoder.fit(labels.to_numpy())

    print(
        f"Fitted LabelEncoder on {labels.len()} '{split_name}' rows; {len(encoder.classes_)} classes:"
    )
    for class_id, class_name in enumerate(encoder.classes_):
        print(f"  {class_id:>2} -> {class_name}")
    return encoder

In [ ]:
def dump_label_encoder(encoder: LabelEncoder, output_path: str = PATH_LABEL_ENCODER):
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    joblib.dump(encoder, output_path)
    print(f"Saved fitted LabelEncoder to: {output_path}")

In [ ]:
def get_label_encoder(
    labels_folder_name: str, output_path: str = PATH_LABEL_ENCODER
) -> LabelEncoder:
    encoder = create_label_encoder(labels_folder_name)
    dump_label_encoder(encoder, output_path)
    return encoder

In [ ]:
label_encoder = get_label_encoder(PATH_FOLDER_SPLIT_LABEL)

### Encode Labels

The fitted encoder is applied to every split, writing one integer-label Parquet file per input file into `PATH_FOLDER_ENCODED_LABEL/{train,dev,test}`. The `label` column name is kept unchanged, so the encoded folder is a drop-in replacement for `PATH_FOLDER_SPLIT_LABEL` in the downstream data-loading helpers.

In [100]:
def load_label_encoder(path: str = PATH_LABEL_ENCODER) -> LabelEncoder:
    return joblib.load(path)

In [101]:
def _encode_single_label_file(
    file_path: str, encoder: LabelEncoder, output_split_folder: str
):
    label_column = LABEL_COLUMN.lower()
    labels = pd.read_parquet(file_path)[label_column]

    encoded_df = pd.DataFrame({label_column: encoder.transform(labels.to_numpy())})
    encoded_df.to_parquet(
        os.path.join(output_split_folder, os.path.basename(file_path)),
        engine="pyarrow",
        compression="snappy",
        index=False,
    )


In [102]:
def encode_split_labels(
    labels_folder_name: str, encoder_path: str, output_folder_name: str
):
    encoder = load_label_encoder(encoder_path)

    for split_name in ("train", "dev", "test"):
        file_paths = get_split_parquet_files(labels_folder_name, split_name)
        output_split_folder = os.path.join(output_folder_name, split_name)
        os.makedirs(output_split_folder, exist_ok=True)
        total = len(file_paths)

        with parallel_config(backend="loky", inner_max_num_threads=1, verbose=0):
            Parallel(n_jobs=-1, verbose=5)(
                delayed(_encode_single_label_file)(
                    file_path, encoder, output_split_folder
                )
                for file_path in file_paths
            )

        print(f"Finished encoding {total} {split_name} label files.")

In [103]:
encode_split_labels(PATH_FOLDER_SPLIT_LABEL, PATH_LABEL_ENCODER, PATH_FOLDER_ENCODED_LABEL)

Found 168 train files


/home/joshuans/miniforge3/envs/rapids/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed:    2.6s


Finished encoding 168 train label files.
Found 168 dev files
Finished encoding 168 dev label files.
Found 168 test files
Finished encoding 168 test label files.


[Parallel(n_jobs=-1)]: Done 168 out of 168 | elapsed:    2.8s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 155 out of 168 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done 168 out of 168 | elapsed:    0.1s finished
[Parallel(n_jobs=-1)]: Using backend LokyBackend with 24 concurrent workers.
[Parallel(n_jobs=-1)]: Done  24 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 155 out of 168 | elapsed:    0.1s remaining:    0.0s
[Parallel(n_jobs=-1)]: Done 168 out of 168 | elapsed:    0.1s finished


### Decode Predictions

`decode_labels` loads the pickled encoder and maps integer class ids (model predictions, or any encoded label column) back to their original human-readable strings.

In [97]:
def decode_labels(encoded, encoder_path: str = PATH_LABEL_ENCODER) -> np.ndarray:
    encoder = load_label_encoder(encoder_path)
    return encoder.inverse_transform(np.asarray(encoded))

## 2.7. Normalized or Feature Scaling

Features are standardized (zero mean, unit variance) using `StandardScaler`. As with imputation, the scaler is fit with `partial_fit` on the **train split only** (processed file-by-file to avoid loading the full ~16M-row dataset into memory at once), then reused to transform train/dev/test. This keeps dev/test statistically unseen during fitting and puts every feature on a comparable scale, which PCA and distance/gradient-based models both depend on.

### Create Scaler

In [ ]:
def fit_scaler_on_files(
    scaler: StandardScaler, file_paths: list[str]
) -> StandardScaler:
    total = len(file_paths)
    for i, file_path in enumerate(file_paths, start=1):
        print(f"Processing [{i}/{total}] {file_path}...", end="\r", flush=True)
        df = pd.read_parquet(file_path)

        if df.empty:
            print(f"Empty file found: {file_path}")
            continue

        scaler.partial_fit(df)
    return scaler

In [ ]:
def create_standard_scaler(
    source_folder_name: str, split_name: str = "train"
) -> StandardScaler:
    file_paths = get_split_parquet_files(source_folder_name, split_name)
    scaler = StandardScaler()
    scaler = fit_scaler_on_files(scaler, file_paths)

    if not hasattr(scaler, "mean_"):
        raise ValueError("The scaler could not be fitted because all files were empty.")

    print(" " * 100, end="\r")
    print(
        f"Successfully fitted scaler using {len(file_paths)} Parquet file(s) from '{split_name}'."
    )

    return scaler

In [ ]:
def dump_standard_scaler(scaler: StandardScaler, output_path: str):
    joblib.dump(scaler, output_path)

In [ ]:
def get_standard_scaler(source_folder_name: str, output_path: str):
    scaler = create_standard_scaler(source_folder_name)
    dump_standard_scaler(scaler, output_path)

In [ ]:
get_standard_scaler(PATH_FOLDER_IMPUTED, PATH_SCALER)

### Scale Training Data

In [ ]:
def load_standard_scaler(path: str) -> StandardScaler:
    return joblib.load(path)

In [ ]:
def _scale_single_file(
    file_path: str, scaler: StandardScaler, output_split_folder: str
):
    df = pd.read_parquet(file_path)
    columns = df.columns.tolist()

    data = scaler.transform(df).astype(PARQUET_FLOAT_DTYPE, copy=False)
    scaled_df = pd.DataFrame(data, columns=columns)

    scaled_df.to_parquet(
        os.path.join(output_split_folder, os.path.basename(file_path)),
        engine="pyarrow",
        compression="snappy",
        index=False,
    )

In [ ]:
def scale_split_files(
    source_folder_name: str, scaler_path: str, output_folder_name: str
):
    scaler = load_standard_scaler(scaler_path)

    for split_name in ("train", "dev", "test"):
        file_paths = get_split_parquet_files(source_folder_name, split_name)
        output_split_folder = os.path.join(output_folder_name, split_name)
        os.makedirs(output_split_folder, exist_ok=True)
        total = len(file_paths)

        with parallel_config(backend="loky", inner_max_num_threads=1):
            Parallel(n_jobs=-1, verbose=5)(
                delayed(_scale_single_file)(file_path, scaler, output_split_folder)
                for file_path in file_paths
            )

        print(f"Finished scaling {total} {split_name} files.")

In [ ]:
scale_split_files(PATH_FOLDER_IMPUTED, PATH_SCALER, PATH_FOLDER_SCALED)

## 2.8. Principal Component Analysis (PCA)

`IncrementalPCA` is used instead of standard `PCA` because it supports `partial_fit` on mini-batches, so the scaled dataset never needs to be fully loaded into memory. Transformers are fit for a range of component counts (5-40) so the explained-variance-vs-components trade-off can be inspected before committing to a final value.

### Creating Incremental Principal Component Analysis

In [ ]:
LIST_PC_COMPONENTS = [5,10,15,20,25,30,35,40]

In [ ]:
def fit_ipca_on_files(ipca: IncrementalPCA, file_paths: list[str]) -> IncrementalPCA:
    total = len(file_paths)
    for i, file_path in enumerate(file_paths, start=1):
        print(f"Processing [{i}/{total}] {file_path}...", end="\r", flush=True)
        df = pd.read_parquet(file_path)

        if df.empty:
            print(f"Empty file found: {file_path}")
            continue

        ipca.partial_fit(df)
    print(" " * 100, end="\r")
    return ipca

In [ ]:
def create_ipca(
    n_components, source_folder_name: str, split_name: str = "train"
) -> IncrementalPCA:
    print(f"Creating IPCA transfromer with {n_components} pc")
    file_paths = get_split_parquet_files(source_folder_name, split_name)
    ipca = IncrementalPCA(n_components=n_components)
    ipca = fit_ipca_on_files(ipca, file_paths)
    print(f"Successfully fitted ipca with {n_components} pc from '{split_name}'.")

    return ipca

In [ ]:
def get_ipca_transformer_path(pc: int, ipca_path: str) -> str:
    return f"{pc}-pc-{ipca_path}"

In [ ]:
def dump_ipca(
    ipca: IncrementalPCA,
    transformer_folder: str = PATH_FOLDER_IPCA_TRANSFORMER,
    ipca_path: str = PATH_IPCA,
):
    filename = get_ipca_transformer_path(ipca.n_components_, ipca_path)
    joblib.dump(ipca, os.path.join(transformer_folder, filename))

In [ ]:
def get_ipca(list_pc_components: list, source_folder_name: str, output_path: str):
    os.makedirs(PATH_FOLDER_IPCA_TRANSFORMER, exist_ok=True)
    for pc in list_pc_components:
        ipca = create_ipca(pc, source_folder_name)
        dump_ipca(ipca, PATH_FOLDER_IPCA_TRANSFORMER, output_path)

In [ ]:
get_ipca(LIST_PC_COMPONENTS,PATH_FOLDER_SCALED, PATH_IPCA)

### Evaluate Incremental Principal Component Analysis

In [ ]:
def load_ipca(
    pc: int,
    transformer_folder: str = PATH_FOLDER_IPCA_TRANSFORMER,
    ipca_path: str = PATH_IPCA,
) -> IncrementalPCA:
    filename = get_ipca_transformer_path(pc, ipca_path)
    return joblib.load(os.path.join(transformer_folder, filename))

In [ ]:
def evaluate_ipca_variance(list_pc_components: list) -> pd.DataFrame:
    rows = []
    for pc in list_pc_components:
        ipca = load_ipca(pc)

        cum_var = np.cumsum(ipca.explained_variance_ratio_)
        rows.append(
            {
                "n_components": pc,
                "total_explained_variance": cum_var[-1],
                "explained_variance_ratio": ipca.explained_variance_ratio_,
            }
        )

    return pd.DataFrame(rows)

In [ ]:
def plot_ipca_variance(evaluation_results: pd.DataFrame):
    plt.figure(figsize=(8, 5))

    x = evaluation_results["n_components"]
    y = evaluation_results["total_explained_variance"]
    plt.plot(x, y, marker="o", color="black")
    for xi, yi in zip(x, y):
        plt.annotate(
            f"{yi:.3f}",
            (xi, yi),
            xytext=(0, 8),
            textcoords="offset points",
            ha="center",
        )

    plt.axhline(0.95, color="red", linestyle="--", label="95% threshold")
    plt.xlabel("Principal Components")
    plt.ylabel("Cumulative Explained Variance")
    plt.title("IPCA: Variance Retained vs. Principal Components")
    plt.legend()
    plt.grid(True)

    plt.show()

In [ ]:
ipca_variance_results = evaluate_ipca_variance(LIST_PC_COMPONENTS)
plot_ipca_variance(ipca_variance_results)

### Implement Incremental Principal Component Analysis

`IPCA_PC_USED = 25` was chosen from the variance plot above as the smallest evaluated component count that clears the 95% cumulative explained-variance threshold.

In [ ]:
IPCA_PC_USED = 25

In [ ]:
def _ipca_transform_single_file(
    file_path: str, ipca: IncrementalPCA, output_split_folder: str
):
    df = pd.read_parquet(file_path)

    data = ipca.transform(df).astype(PARQUET_FLOAT_DTYPE, copy=False)
    pc_columns = [f"pc{i+1}" for i in range(data.shape[1])]
    scaled_df = pd.DataFrame(data, columns=pc_columns)

    scaled_df.to_parquet(
        os.path.join(output_split_folder, os.path.basename(file_path)),
        engine="pyarrow",
        compression="snappy",
        index=False,
    )

In [ ]:
def ipca_split_files(source_folder_name: str, ipca_path: str, output_folder_name: str):
    ipca = load_ipca(IPCA_PC_USED)

    for split_name in ("train", "dev", "test"):
        file_paths = get_split_parquet_files(source_folder_name, split_name)
        output_split_folder = os.path.join(output_folder_name, split_name)
        os.makedirs(output_split_folder, exist_ok=True)
        total = len(file_paths)

        with parallel_config(backend="loky", inner_max_num_threads=1):
            Parallel(n_jobs=-1, verbose=5)(
                delayed(_ipca_transform_single_file)(
                    file_path, ipca, output_split_folder
                )
                for file_path in file_paths
            )

        print(f"Finished scaling {total} {split_name} files.")

In [ ]:
ipca_split_files(PATH_FOLDER_SCALED, PATH_IPCA, PATH_FOLDER_IPCA)

## 2.9. Synthetic Minor Oversampling Technique

CSE-CIC-IDS2018 is heavily imbalanced (Benign traffic dominates; some attack classes are a small fraction of a percent). SMOTE is applied to the **training split only** (never dev/test, so evaluation still reflects real-world class balance) to synthesize minority-class samples and reduce the model's bias toward the majority class.

SMOTE resamples the **PCA-reduced train features** (`PATH_FOLDER_IPCA`, paired with the original `PATH_FOLDER_SPLIT_LABEL` labels), since that's the final feature representation a downstream model would train on. Fitting concatenates every train file via polars (`load_train_features_and_labels`), since imbalanced-learn needs the full split in memory to compute global class counts. Once fitted, the transformer is applied **one file at a time** (`load_smote`, `resample_with_smote`, `save_smote_train_data`) using pandas instead - matching the streaming approach used for IPCA/normalization elsewhere - so resampling never needs the whole split in memory at once; imbalanced-learn only accepts numpy arrays, so the conversion happens right at the `fit_resample` call and the result is wrapped back into a `pd.DataFrame`/`pd.Series`.

Because some attack classes (e.g. SQL Injection) have very few samples in this dataset, `fit_smote` clamps `k_neighbors` down to what the smallest class can support instead of letting `fit_resample` raise on the default `k_neighbors=5`. Fitting is kept separate from resampling: `fit_smote` fits and `dump_smote` persists the fitted transformer via joblib right away, and `resample_with_smote` performs the actual oversampling per file afterwards. Two things can still go wrong at the per-file level that the global fit can't see: (1) a class that's fine globally can be scarce in one specific chunk (e.g. one file has only 5 `Benign` rows against 59,995 `DDOS attack-HOIC` rows), so `resample_with_smote` re-checks `k_neighbors` against that file's own smallest class and clamps further if needed; (2) since each day's capture is chunked in original time order, most individual chunks fall entirely outside the attack window and end up **100% Benign** (123 of the 168 train files, in practice) - `fit_resample` requires at least 2 classes, so `resample_with_smote` detects a single-class file and passes it through unresampled rather than erroring.

### Create SMOTE

In [ ]:
def load_train_features_and_labels(
    features_folder_name: str, labels_folder_name: str, split_name: str = "train"
) -> tuple[pl.DataFrame, pl.Series]:
    feature_files = get_split_parquet_files(features_folder_name, split_name)
    label_files = get_split_parquet_files(labels_folder_name, split_name)

    print(
        f"Loading {len(feature_files)} '{split_name}' feature file(s)...",
        end="",
        flush=True,
    )
    features_df = pl.concat([pl.scan_parquet(f) for f in feature_files]).collect()
    labels_df = (
        pl.concat([pl.scan_parquet(f) for f in label_files])
        .select(LABEL_COLUMN.lower())
        .collect()
    )

    print(f"Loaded {features_df.height} rows for '{split_name}'.")
    return features_df, labels_df.to_series()

In [ ]:
def get_class_count(labels: pl.Series) -> dict:
    counts = labels.value_counts()
    return dict(zip(counts[labels.name].to_list(), counts["count"].to_list()))

In [ ]:
def fit_smote(
    features: pl.DataFrame, labels: pl.Series, random_state: int = 42
) -> SMOTE:
    class_counts = get_class_count(labels)
    smallest_class_count = min(class_counts.values())
    k_neighbors = max(1, min(5, smallest_class_count - 1))

    if k_neighbors < 5:
        print(
            f"Warning: smallest class has {smallest_class_count} samples; reducing k_neighbors to {k_neighbors}"
        )

    smote = SMOTE(random_state=random_state, k_neighbors=k_neighbors)
    with parallel_config(n_jobs=-1):
        smote.fit(features.to_numpy(), labels.to_numpy())
    return smote

In [ ]:
def dump_smote(
    smote: SMOTE,
    transformer_folder: str = PATH_FOLDER_SMOTE_TRANSFORMER,
    smote_path: str = PATH_SMOTE,
):
    os.makedirs(transformer_folder, exist_ok=True)
    output_path = os.path.join(transformer_folder, smote_path)
    joblib.dump(smote, output_path)
    print(f"Saved fitted SMOTE transformer to: {output_path}")

In [ ]:
def get_smote_transformer(features_folder_name: str, labels_folder_name: str):
    features, labels = load_train_features_and_labels(
        features_folder_name, labels_folder_name
    )
    smote = fit_smote(features, labels)
    dump_smote(smote)

In [ ]:
get_smote_transformer(PATH_FOLDER_IPCA, PATH_FOLDER_ENCODED_LABEL)

### Implement SMOTE

In [ ]:
def load_smote(
    transformer_folder: str = PATH_FOLDER_SMOTE_TRANSFORMER,
    smote_path: str = PATH_SMOTE,
) -> SMOTE:
    return joblib.load(os.path.join(transformer_folder, smote_path))

In [ ]:
def resample_with_smote(
    smote: SMOTE, features: pd.DataFrame, labels: pd.Series
) -> tuple[pd.DataFrame, pd.Series]:
    if labels.nunique() < 2:
        return features, labels

    smallest_class_count = labels.value_counts().min()
    k_neighbors = max(1, min(smote.k_neighbors, smallest_class_count - 1))

    if k_neighbors < smote.k_neighbors:
        print(
            f"Warning: smallest class in this file has {smallest_class_count} samples; reducing k_neighbors to {k_neighbors}"
        )
        smote = SMOTE(
            random_state=smote.random_state,
            k_neighbors=k_neighbors,
            sampling_strategy=smote.sampling_strategy,
        )

    features_resampled, labels_resampled = cast(
        tuple[np.ndarray, np.ndarray],
        smote.fit_resample(features.to_numpy(), labels.to_numpy()),
    )
    resampled_features = pd.DataFrame(
        features_resampled, columns=features.columns
    ).astype(PARQUET_FLOAT_DTYPE)
    return resampled_features, pd.Series(labels_resampled, name=labels.name)

In [ ]:
def save_smote_train_data(
    features: pd.DataFrame,
    labels: pd.Series,
    file_name: str,
    output_folder_name: str = PATH_FOLDER_SMOTE,
):
    output_split_folder = os.path.join(output_folder_name, "train")
    os.makedirs(output_split_folder, exist_ok=True)

    output_df = features.assign(**{LABEL_COLUMN.lower(): labels.to_numpy()})
    output_df.to_parquet(
        os.path.join(output_split_folder, file_name),
        engine="pyarrow",
        compression="snappy",
        index=False,
    )

In [ ]:
def _smote_resample_single_file(
    feature_file_path: str, label_file_path: str, smote: SMOTE, output_folder_name: str
):
    features = pd.read_parquet(feature_file_path)
    labels = pd.read_parquet(label_file_path)[LABEL_COLUMN.lower()]

    features_resampled, labels_resampled = resample_with_smote(smote, features, labels)
    save_smote_train_data(
        features_resampled,
        labels_resampled,
        os.path.basename(feature_file_path),
        output_folder_name,
    )

In [ ]:
def smote_resample_files(
    features_folder_name: str,
    labels_folder_name: str,
    output_folder_name: str = PATH_FOLDER_SMOTE,
):
    smote = load_smote()

    feature_files = get_split_parquet_files(features_folder_name, "train")
    label_files = get_split_parquet_files(labels_folder_name, "train")
    total = len(feature_files)

    with parallel_config(backend="loky", inner_max_num_threads=1, verbose=0):
        Parallel(n_jobs=-1, verbose=5)(
            delayed(_smote_resample_single_file)(
                feature_file_path, label_file_path, smote, output_folder_name
            )
            for feature_file_path, label_file_path in zip(feature_files, label_files)
        )

    print(f"Finished SMOTE-resampling {total} train file(s).")

In [ ]:
smote_resample_files(PATH_FOLDER_IPCA, PATH_FOLDER_ENCODED_LABEL, PATH_FOLDER_SMOTE)

### Evaluate SMOTE

In [ ]:
def get_class_count_from_label(source_folder_path: str):
    label_files = get_split_parquet_files(source_folder_path, "train")
    labels = (
        pl.concat([pl.scan_parquet(f) for f in label_files])
        .select(LABEL_COLUMN.lower())
        .collect()
    )
    return get_class_count(labels.to_series()), len(labels)

In [ ]:
before_class_count, before_row_count = get_class_count_from_label(
    PATH_FOLDER_ENCODED_LABEL
)
print(f"Row count: {before_row_count}")
print(f"Class count:\n{json.dumps(before_class_count, indent=4)}")

In [ ]:
after_class_count, after_row_count = get_class_count_from_label(PATH_FOLDER_SMOTE)
print(f"Row count: {after_row_count}")
print(f"Class count:\n{json.dumps(after_class_count, indent=4)}")

In [ ]:
def plot_smote_comparison(
    before_class_count: dict[str, int],
    after_class_count: dict[str, int],
) -> None:
    classes = sorted(set(before_class_count) | set(after_class_count))
    before = []
    for class_name in classes:
        before.append(before_class_count.get(class_name, 0))

    after = []
    for class_name in classes:
        after.append(after_class_count.get(class_name, 0))

    df = pd.DataFrame(
        {
            "Class": classes,
            "Before SMOTE": before,
            "After SMOTE": after,
        }
    )

    ax = df.set_index("Class").plot(
        kind="bar",
        figsize=(14, 7),
        width=0.8,
    )

    ax.set_title("Class Distribution Before and After SMOTE")
    ax.set_xlabel("Class")
    ax.set_ylabel("Number of Samples")

    plt.xticks(rotation=45, ha="right")
    plt.legend(title="Dataset")
    plt.tight_layout()
    plt.show()

In [ ]:
plot_smote_comparison(before_class_count, after_class_count)

# 3. Load Datasets

## Load Training Data

In [40]:
train_x, train_y = get_polars_data_frame_with_label(PATH_FOLDER_SMOTE)

Found 168 train files


## Load Validation Data

In [41]:
dev_x = get_polars_data_frame_without_label(PATH_FOLDER_IPCA,"dev")
dev_y = get_polars_data_frame_without_label(PATH_FOLDER_ENCODED_LABEL,"dev").to_series()

Found 168 dev files
Found 168 dev files] data-ipca/dev/2018-03-02-Friday_TrafficForML_CICFlowMeter_00011.parquet......


## Load Test Data

In [68]:
test_x = get_polars_data_frame_without_label(PATH_FOLDER_IPCA,"test")
test_y = get_polars_data_frame_without_label(PATH_FOLDER_ENCODED_LABEL,"test").to_series()

Found 168 test files
Found 168 test files data-ipca/test/2018-03-02-Friday_TrafficForML_CICFlowMeter_00011.parquet......


# 4. K-Nearest Neighbor

In [35]:
def get_knn_name(n,metric,weights):
    return f"knn-k-{n}-m-{metric}-w-{weights}.pkl"

## Shared Helpers for Model Selection

Both hyperparameter searches below need the same conversion helpers, the same
subsampler and the same `SEARCH_DEV_MIN_PER_CLASS` floor, so all of it lives here
rather than being duplicated per model. Anything a search reads as a *default argument*
in particular has to be defined above both searches: a default is evaluated when the
`def` cell runs, so a constant parked in the KNN section makes the SVM search fail with
a `NameError` unless the KNN section happens to have been run first.

`stratified_subsample` exists because neither model can be searched over the full
SMOTE'd train split. A brute-force KNN query costs `O(n_query * n_train)` and kernel
SVM training is between `O(n^2)` and `O(n^3)` in the number of training rows, so at
~11M train rows a single hyperparameter combination would take longer than the whole
study. Sampling *proportionally per class*, with a floor of `min_per_class` rows,
keeps the rare attack classes (e.g. SQL Injection) present in the search sample
instead of letting uniform random sampling drop them entirely.

The floor matters more than it looks. Purely proportional sampling gives a class that
holds 3 of ~100k dev rows exactly *one* row in the subsample, and macro F1 - the
selection criterion - then swings by a full 1/n_classes depending on whether that
single row happens to be classified correctly. Both searches therefore pass
`SEARCH_DEV_MIN_PER_CLASS` when subsampling dev, so every class contributes a stable
per-class score rather than a coin flip.

Note that the subsample is only used for **model selection**. The final chosen model
is refit and every reported metric is computed on the full, untouched dev/test splits,
so the subsampling never enters the numbers that go into the results chapter.

In [42]:
def to_features(df) -> np.ndarray:
    """polars DataFrame -> C-contiguous float32 array for cuML."""
    if isinstance(df, (pl.DataFrame, pl.Series)):
        df = df.to_numpy()
    return np.ascontiguousarray(df, dtype=np.float32)


def to_labels(s) -> np.ndarray:
    """polars Series -> int32 labels (cuML classifiers reject float labels)."""
    if isinstance(s, (pl.DataFrame, pl.Series)):
        s = s.to_numpy()
    return np.asarray(s).ravel().astype(np.int32)

In [43]:
def to_numpy(a):
    if hasattr(a, "to_numpy"):
        return np.asarray(a.to_numpy()).ravel()
    if hasattr(a, "get"):
        return a.get().ravel()
    return np.asarray(a).ravel()

In [44]:
def to_numpy_2d(a) -> np.ndarray:
    """cupy / cuDF / polars 2-D result -> 2-D numpy array.

    `to_numpy` above ravels its input, which is correct for a prediction vector but
    destroys the (n_query, k) shape of a `kneighbors` result.
    """
    if hasattr(a, "to_numpy"):
        return np.asarray(a.to_numpy())
    if hasattr(a, "get"):
        return np.asarray(a.get())
    return np.asarray(a)

In [45]:
# Shared by every hyperparameter search below, not just KNN: `search_svm` reads it
# as a default argument, which is evaluated when the cell defining it runs. Keeping
# it in the KNN section made the SVM search a NameError unless KNN had been run first.
SEARCH_DEV_MIN_PER_CLASS = 200

In [46]:
def stratified_subsample(
    x,
    y,
    n_samples: int | None,
    random_state: int = 42,
    min_per_class: int = 1,
) -> tuple[np.ndarray, np.ndarray]:
    """Class-proportional row subsample as (float32 features, int32 labels).

    Passing `n_samples=None` (or a value at least as large as the split) returns the
    whole split unchanged, so the same call site works for both search and final fit.
    """
    labels = to_labels(y)
    total = labels.shape[0]

    if n_samples is None or n_samples >= total:
        return to_features(x), labels

    rng = np.random.default_rng(random_state)
    classes, counts = np.unique(labels, return_counts=True)

    quota = np.floor(counts / total * n_samples).astype(np.int64)
    quota = np.maximum(quota, np.minimum(min_per_class, counts))
    quota = np.minimum(quota, counts)

    selected = []
    for class_id, class_quota in zip(classes, quota):
        class_index = np.flatnonzero(labels == class_id)
        if class_quota < class_index.size:
            class_index = rng.choice(class_index, size=int(class_quota), replace=False)
        selected.append(class_index)

    selected = np.sort(np.concatenate(selected))

    # A boolean mask keeps this working for both polars and numpy inputs, and
    # filtering (rather than gathering) preserves the original row order.
    mask = np.zeros(total, dtype=bool)
    mask[selected] = True
    subset = x.filter(pl.Series(mask)) if isinstance(x, pl.DataFrame) else x[mask]

    print(
        f"Subsampled {total:,} -> {int(mask.sum()):,} rows across {classes.size} classes"
    )
    return to_features(subset), labels[mask]

In [47]:
def dump_search_results(
    results: pd.DataFrame, name: str, folder: str = PATH_FOLDER_SEARCH_RESULT
) -> str:
    os.makedirs(folder, exist_ok=True)
    file_path = os.path.join(folder, name)
    results.to_csv(file_path, index=False)
    print(f"Saved search results to: {file_path}")
    return file_path

In [48]:
def get_best_hyperparameter(
    results: pd.DataFrame, score_column: str = "f1_macro"
) -> dict:
    """Pick the top row by `score_column`.

    macro F1 is the selection criterion rather than accuracy: the dev split keeps its
    real class balance (Benign dominates), so accuracy would be maximised by a model
    that predicts the rare attack classes badly.
    """
    best = results.sort_values(score_column, ascending=False).iloc[0]
    print(f"Best by {score_column}:")
    for key, value in best.items():
        print(f"  {key:>16} = {value}")
    return best.to_dict()

## Hyperparameter Search for KNN

Uses nearest neighbors

In [ ]:
list_of_n_components = [i for i in range(1, 15, 2)]
list_of_distance_metrics = ["manhattan", "euclidean", "cosine"]
list_of_weighing = ["uniform", "distance"]

The grid is `k x metric x weighting` = 7 x 3 x 2 = 42 combinations. Fitting a KNN is
cheap (it only builds an index); the cost is entirely in the dev-set query, so a naive
loop would run 42 full queries.

`search_knn` runs **one query per metric** instead. For a fixed metric, `kneighbors`
returns neighbours already sorted by distance, so the k nearest for any smaller k are
just the leading columns of the same result, and both weighting schemes are re-votes
over those same columns. That is 3 queries instead of 42 for identical results.

The vote itself is done with `np.bincount` over a flattened `(row, class)` index rather
than a Python loop over query rows, which is what makes re-voting 42 times essentially
free compared to the query.

In [ ]:
KNN_SEARCH_TRAIN_SAMPLES = 500_000
KNN_SEARCH_DEV_SAMPLES = 200_000
KNN_SEARCH_QUERY_CHUNK = 20_000
KNN_SEARCH_RANDOM_STATE = 42

In [ ]:
def _knn_vote(
    neighbor_labels: np.ndarray,
    neighbor_distances: np.ndarray,
    n_classes: int,
    weighting: str,
) -> np.ndarray:
    """Majority vote over pre-computed neighbours.

    `neighbor_labels` and `neighbor_distances` are both (n_query, k).
    """
    n_query, n_neighbors = neighbor_labels.shape

    if weighting == "uniform":
        weights = np.ones((n_query, n_neighbors), dtype=np.float64)
    else:
        with np.errstate(divide="ignore"):
            weights = 1.0 / neighbor_distances.astype(np.float64)
        # A query point that coincides exactly with a training point gets an infinite
        # weight. sklearn's rule is that only the zero-distance neighbours vote in
        # that row, which matters here because SMOTE interpolates new points onto the
        # segments between existing ones and produces exact duplicates.
        exact = ~np.isfinite(weights)
        exact_rows = exact.any(axis=1)
        if exact_rows.any():
            weights[exact_rows] = exact[exact_rows].astype(np.float64)

    row_offset = np.arange(n_query, dtype=np.int64)[:, None] * n_classes
    flat_index = (row_offset + neighbor_labels).ravel()
    scores = np.bincount(
        flat_index, weights=weights.ravel(), minlength=n_query * n_classes
    )
    return scores.reshape(n_query, n_classes).argmax(axis=1).astype(np.int32)

In [ ]:
def _kneighbors_by_chunk(
    index, query: np.ndarray, n_neighbors: int, chunk_size: int
) -> tuple[np.ndarray, np.ndarray]:
    total = query.shape[0]
    n_chunks = math.ceil(total / chunk_size)
    distances = []
    indices = []
    started = time.time()

    for i, start in enumerate(range(0, total, chunk_size), start=1):
        chunk_distance, chunk_index = index.kneighbors(
            query[start : start + chunk_size], n_neighbors=n_neighbors
        )
        distances.append(to_numpy_2d(chunk_distance).astype(np.float32, copy=False))
        indices.append(to_numpy_2d(chunk_index).astype(np.int64, copy=False))

        elapsed = time.time() - started
        eta = elapsed / i * (n_chunks - i)
        print(
            f"    neighbours [{i}/{n_chunks}] elapsed {elapsed:,.0f}s eta {eta:,.0f}s",
            end="\r",
            flush=True,
        )

    print(" " * 100, end="\r")
    return np.vstack(distances), np.vstack(indices)

In [ ]:
def search_knn(
    train_x,
    train_y,
    dev_x,
    dev_y,
    list_n_neighbors: list[int],
    list_distance_metrics: list[str],
    list_weighting: list[str],
    train_samples: int | None = KNN_SEARCH_TRAIN_SAMPLES,
    dev_samples: int | None = KNN_SEARCH_DEV_SAMPLES,
    dev_min_per_class: int = SEARCH_DEV_MIN_PER_CLASS,
    chunk_size: int = KNN_SEARCH_QUERY_CHUNK,
    random_state: int = KNN_SEARCH_RANDOM_STATE,
) -> pd.DataFrame:
    print("Preparing search subsamples...")
    search_train_x, search_train_y = stratified_subsample(
        train_x, train_y, train_samples, random_state
    )
    search_dev_x, search_dev_y = stratified_subsample(
        dev_x, dev_y, dev_samples, random_state, min_per_class=dev_min_per_class
    )

    n_classes = int(max(search_train_y.max(), search_dev_y.max())) + 1
    max_neighbors = max(list_n_neighbors)
    rows = []

    for distance_metric in list_distance_metrics:
        print(f"Fitting neighbour index (metric={distance_metric})...")
        index = NearestNeighbors(n_neighbors=max_neighbors, metric=distance_metric)
        index.fit(search_train_x)

        neighbor_distances, neighbor_indices = _kneighbors_by_chunk(
            index, search_dev_x, max_neighbors, chunk_size
        )
        neighbor_labels = search_train_y[neighbor_indices]

        for n_neighbors in list_n_neighbors:
            for weighting in list_weighting:
                pred = _knn_vote(
                    neighbor_labels[:, :n_neighbors],
                    neighbor_distances[:, :n_neighbors],
                    n_classes,
                    weighting,
                )
                scores = evaluate(pred=pred, true=search_dev_y).iloc[0].to_dict()
                rows.append(
                    {
                        "n_neighbors": n_neighbors,
                        "metric": distance_metric,
                        "weights": weighting,
                        **scores,
                    }
                )
                print(
                    f"  k={n_neighbors:>3} metric={distance_metric:<10}"
                    f" weights={weighting:<8} f1_macro={scores['f1_macro']:.4f}"
                )

        del index, neighbor_distances, neighbor_indices, neighbor_labels

    return pd.DataFrame(rows).sort_values(
        "f1_macro", ascending=False, ignore_index=True
    )

In [ ]:
knn_search_results = search_knn(
    train_x,
    train_y,
    dev_x,
    dev_y,
    list_of_n_components,
    list_of_distance_metrics,
    list_of_weighing,
)
dump_search_results(knn_search_results, "knn-search.csv")
display(knn_search_results)

In [ ]:
best_knn = get_best_hyperparameter(knn_search_results)
KNN_BEST_N_COMPONENT = int(best_knn["n_neighbors"])
KNN_BEST_DISTANCE_METRIC = str(best_knn["metric"])
KNN_BEST_WEIGHING_METHOD = str(best_knn["weights"])

The `KNN_MODEL_*` constants in the *Evaluate KNN* section further down are still
hardcoded to `k=11 / manhattan / uniform`. Once this search has been run, set them from
`KNN_BEST_*` (and retrain / clear the cached prediction chunks) so the reported model is
the one the search actually selected.

## Train KNN

In [ ]:
def train_knn(
    train_x, train_y, n_neighbors, distance_metric, weighting
) -> KNeighborsClassifier:
    model = KNeighborsClassifier(
        n_neighbors=n_neighbors,
        metric=distance_metric,
        weights=weighting,
    )
    model.fit(to_features(train_x), to_labels(train_y))
    dump_trained_model(
        model, get_knn_name(n_neighbors, distance_metric, weighting), "KNN"
    )

In [ ]:
train_knn(train_x,train_y,11,"manhattan","uniform")

## Evaluate KNN

In [63]:
PREDICT_BY_CHUNK = 10_000
KNN_MODEL_N_COMPONENT = 11
KNN_MODEL_WEIGHING_METHOD = "uniform"
KNN_MODEL_DISTANCE_METRIC = "manhattan"

In [ ]:
def load_knn_model(
    model_path: str, folder_path: str = PATH_FOLDER_MODEL
) -> KNeighborsClassifier:
    return joblib.load(os.path.join(folder_path, "KNN", model_path))

In [ ]:
def knn_predict(
    model_path,
    x,
    chunk_size: int = PREDICT_BY_CHUNK,
    cache_dir: str = PATH_FOLDER_PREDICTION_RESULT,
    cache_dir_subfolder: str = "",
):
    if not cache_dir:
        cache_dir = os.path.join(cache_dir, "KNN")
    if cache_dir_subfolder:
        cache_dir = os.path.join(cache_dir, cache_dir_subfolder)
    os.makedirs(cache_dir, exist_ok=True)

    model = load_knn_model(model_path)

    total = len(x)
    if total == 0:
        return np.empty(0)

    n_chunks = math.ceil(total / chunk_size)
    paths = []

    for i, start in enumerate(range(0, total, chunk_size), start=1):
        path = os.path.join(cache_dir, f"knn_{i:05d}.npy")
        paths.append(path)

        if os.path.exists(path):
            print(f"Cached [{i}/{n_chunks}]", end="\r", flush=True)
            continue

        if isinstance(x, (pl.DataFrame, pl.Series)):
            chunk = x.slice(start, chunk_size)
        else:
            chunk = x[start : start + chunk_size]

        pred = to_numpy(model.predict(to_features(chunk)))

        tmp = path + ".tmp"
        with open(tmp, "wb") as f:
            np.save(f, pred)
        os.replace(tmp, path)

        done = min(start + chunk_size, total)
        print(
            f"Predicting [{i}/{n_chunks}] {done:,}/{total:,} rows ",
            end="\r",
            flush=True,
        )

    print(f"Predicted {total:,} rows" + " " * 20)
    return np.concatenate([np.load(p) for p in paths])

In [ ]:
knn_predict(get_knn_name(KNN_MODEL_N_COMPONENT,KNN_MODEL_DISTANCE_METRIC,KNN_MODEL_WEIGHING_METHOD),dev_x,cache_dir_subfolder="dev")

In [ ]:
knn_predict(get_knn_name(KNN_MODEL_N_COMPONENT,KNN_MODEL_DISTANCE_METRIC,KNN_MODEL_WEIGHING_METHOD),test_x,cache_dir_subfolder="test")

### Report

In [86]:
def _load_chunks(folder: str) -> np.ndarray:
    if not os.path.isdir(folder):
        raise FileNotFoundError(folder)
    names = sorted(n for n in os.listdir(folder) if n.endswith(".npy"))
    if not names:
        raise FileNotFoundError(f"no .npy chunks in {folder}")
    return np.concatenate([np.load(os.path.join(folder, n)) for n in names])

In [87]:
def load_true_labels(labels_folder: str, split_name: str) -> np.ndarray:
    return to_numpy(
        get_polars_data_frame_without_label(labels_folder, split_name).to_series()
    )

In [88]:
def load_predictions_and_labels(
    folder_predict: str,
    split_name: str,
    labels_folder: str = PATH_FOLDER_ENCODED_LABEL,
) -> tuple[np.ndarray, np.ndarray]:
    pred = _load_chunks(folder_predict)
    true = load_true_labels(labels_folder, split_name)
    if len(pred) != len(true):
        raise ValueError(f"length mismatch: pred={len(pred):,}, true={len(true):,}")
    return pred, true

In [89]:
def get_evaluation_results(
    folder_predict: str,
    split_name: str,
    labels_folder: str = PATH_FOLDER_ENCODED_LABEL,
) -> pd.DataFrame:
    pred, true = load_predictions_and_labels(folder_predict, split_name, labels_folder)
    return evaluate(pred=pred, true=true)

In [90]:
get_evaluation_results(
    os.path.join(PATH_FOLDER_PREDICTION_RESULT, "KNN", "test"),
    "test",
)

Found 168 test files


,accuracy,precision_macro,recall_macro,f1_macro
0,0.980034,0.761029,0.844731,0.74896


In [91]:
knn_test_pred, knn_test_true = load_predictions_and_labels(
    os.path.join(PATH_FOLDER_PREDICTION_RESULT, "KNN", "test"), "test"
)

Found 168 test files


In [92]:
get_classification_report(pred=knn_test_pred, true=knn_test_true)

NameError: name 'decode_labels' is not defined

In [ ]:
cm = get_confusion_matrix(pred=knn_test_pred, true=knn_test_true)
display(cm)

/home/joshuans/miniforge3/envs/rapids/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,Benign,Bot,Brute Force -Web,Brute Force -XSS,DDOS attack-HOIC,DDOS attack-LOIC-UDP,DDoS attacks-LOIC-HTTP,DoS attacks-GoldenEye,DoS attacks-Hulk,DoS attacks-SlowHTTPTest,DoS attacks-Slowloris,FTP-BruteForce,Infilteration,SQL Injection,SSH-Bruteforce
Benign,2688894,8,435,16,2,0,54,19,5,0,10,2,7452,34,19
Bot,12,57226,0,0,0,0,0,0,0,0,0,0,0,0,0
Brute Force -Web,13,0,105,1,0,0,1,0,0,0,0,0,0,2,0
Brute Force -XSS,0,0,1,43,0,0,0,0,0,0,0,0,0,2,0
DDOS attack-HOIC,0,0,0,0,137203,0,0,0,0,0,0,0,0,0,0
DDOS attack-LOIC-UDP,0,0,0,0,0,346,0,0,0,0,0,0,0,0,0
DDoS attacks-LOIC-HTTP,5,0,10,0,0,138,115084,0,0,0,0,0,0,0,0
DoS attacks-GoldenEye,1,0,0,0,0,0,0,8297,3,0,0,0,0,0,0
DoS attacks-Hulk,4,0,0,0,0,0,0,3,92375,0,0,0,0,0,0
DoS attacks-SlowHTTPTest,0,0,0,0,0,0,0,0,0,25,0,27953,0,0,0


# 5. Support Vector Machine

Kernel SVM does not scale to this dataset the way the other classifiers do. Training
is between `O(n^2)` and `O(n^3)` in the number of rows, and prediction costs
`O(n_query * n_support_vectors)`, so the ~11M-row SMOTE'd train split is out of reach
regardless of GPU acceleration.

The SVM is therefore fit on a **class-proportional stratified subsample** of the train
split (`SVM_TRAIN_SAMPLES`), with a smaller subsample again for the hyperparameter
search (`SVM_SEARCH_TRAIN_SAMPLES`). Two points matter for interpreting the results:

1. Only *training* is subsampled. Dev and test predictions are computed over the full
   splits, so SVM metrics are directly comparable with the other four classifiers.
2. The subsample is drawn *after* SMOTE, from an already-balanced split, so
   proportional sampling preserves the balance SMOTE established.

This is a compute constraint rather than a methodological choice, and it should be
stated as such when the SVM results are reported.

Every slow step below reports progress, but by *stage*, not by clock. `SVC.fit` is a
single blocking call with nothing inside it to count, so each stage prints when it
starts and is completed in place when it returns (`step`), while the chunked prediction
loop draws a real bar from its own iterations (`progress_bar`). Neither uses a
background thread.

On the GPU this section uses `cuml.svm.SVC` directly rather than the `cuml.accel`
proxy. The proxy wraps sklearn's `SVC` but its `_gpu_fit` raises `UnsupportedOnGPU`
whenever `y` has more than two classes, so with 15 attack classes every fit silently
fell back to single-core CPU libsvm. `cuml.svm.SVC` routes multiclass through
`cuml.multiclass.MulticlassClassifier` and fits each pairwise binary SVM on the GPU.

In [49]:
def get_svm_name(kernel, c, gamma):
    return f"svm-k-{kernel}-c-{c}-g-{gamma}.pkl"

## Hyperparameter Search for SVM

`gamma` only affects the RBF kernel, so pairing it with `linear` would duplicate every
`C` value for no benefit; `build_svm_grid` pins it to `"scale"` there. With the defaults
below that gives `3 (linear) + 6 (rbf) = 9` combinations.

`fit_seconds` and `n_support` are recorded alongside the scores because they, not the
score, determine whether a combination is affordable at full scale: the support-vector
count sets the per-row cost of predicting the ~3.2M dev and test rows later.

cuML's GPU solver refuses some combinations outright. A high `C` on a hard pairwise
sub-problem raises `Working set has already been initialized!` from its C++ layer -
reproducibly, as the very first fit in a fresh process, with the card at ~106 MiB, so
it is a solver bug and not an out-of-memory condition. Which combination trips it
depends on the subsample.

Those combinations are **skipped and recorded** with NaN scores and the error text
rather than retried on the CPU: libsvm takes over 15 minutes on precisely the
combinations cuML rejects, against roughly 9 seconds for the ones it accepts, so a
retry would cost more than the entire rest of the grid. Check the `error` column
before reading the results - a `NaN` row is a combination that was never scored, not
one that scored badly, and if the grid's best model would have been in that region it
will not appear in the table at all.

In [50]:
list_of_kernels = ["rbf", "linear"]
list_of_c = [0.1, 1.0, 10.0]
list_of_gamma = ["scale", 0.1]

In [51]:
SVM_TRAIN_SAMPLES = 300_000
SVM_SEARCH_TRAIN_SAMPLES = 50_000
SVM_SEARCH_DEV_SAMPLES = 100_000
SVM_RANDOM_STATE = 42
SVM_CACHE_SIZE = 2048
SVM_TOLERANCE = 1e-3
SVM_MAX_ITER = -1

In [52]:
def build_svm_grid(
    kernels: list[str], list_c: list[float], list_gamma: list
) -> list[tuple]:
    combinations = []
    for kernel in kernels:
        for c in list_c:
            if kernel == "linear":
                combinations.append((kernel, c, "scale"))
                continue
            for gamma in list_gamma:
                combinations.append((kernel, c, gamma))
    return combinations

In [53]:
def build_svm(
    kernel: str,
    c: float,
    gamma,
    cache_size: int = SVM_CACHE_SIZE,
    tol: float = SVM_TOLERANCE,
    max_iter: int = SVM_MAX_ITER,
) -> SVC:
    return SVC(
        C=float(c),
        kernel=kernel,
        gamma=gamma,
        cache_size=cache_size,
        tol=tol,
        max_iter=max_iter,
    )

In [54]:
def _support_vector_count(model) -> float:
    """Total support vectors, or NaN when the estimator does not expose them.

    cuML's multiclass wrapper does not always surface `n_support_`, so this must not
    be allowed to break the search loop.
    """
    try:
        n_support = getattr(model, "n_support_", None)
        if n_support is not None:
            return float(np.sum(to_numpy_2d(n_support)))
        support = getattr(model, "support_", None)
        if support is not None:
            return float(np.size(to_numpy_2d(support)))
    except Exception:
        pass
    return float("nan")

In [55]:
def _fit_and_score_svm(
    kernel, c, gamma, train_x, train_y, dev_x, dev_y
) -> tuple[float, float, dict]:
    """Fit one combination, score it on dev, and hand the device memory back."""
    model = build_svm(kernel, c, gamma)

    fit_started = time.time()
    with step(f"fitting on {train_x.shape[0]:,} rows"):
        model.fit(train_x, train_y)
    fit_seconds = time.time() - fit_started

    with step(f"predicting {dev_x.shape[0]:,} dev rows"):
        pred = to_numpy(model.predict(dev_x))

    scores = evaluate(pred=pred, true=dev_y).iloc[0].to_dict()
    n_support = _support_vector_count(model)

    del model
    free_gpu_memory()
    return fit_seconds, n_support, scores

In [56]:
def search_svm(
    train_x,
    train_y,
    dev_x,
    dev_y,
    combinations: list[tuple] | None = None,
    train_samples: int | None = SVM_SEARCH_TRAIN_SAMPLES,
    dev_samples: int | None = SVM_SEARCH_DEV_SAMPLES,
    dev_min_per_class: int = SEARCH_DEV_MIN_PER_CLASS,
    random_state: int = SVM_RANDOM_STATE,
) -> pd.DataFrame:
    if combinations is None:
        combinations = build_svm_grid(list_of_kernels, list_of_c, list_of_gamma)

    total = len(combinations)
    print(f"SVM hyperparameter search over {total} combinations")

    # Start from a clean card: whatever the KNN / IPCA cells above left allocated
    # counts against the same 8 GiB budget this grid needs.
    free_gpu_memory()

    # stratified_subsample prints its own "Subsampled N -> M rows" summary, so these
    # are not wrapped in step() - that would split its line across the " done".
    print("Preparing search subsamples...")
    search_train_x, search_train_y = stratified_subsample(
        train_x, train_y, train_samples, random_state
    )
    search_dev_x, search_dev_y = stratified_subsample(
        dev_x, dev_y, dev_samples, random_state, min_per_class=dev_min_per_class
    )

    n_train, n_features = search_train_x.shape
    n_dev = search_dev_x.shape[0]
    print(f"  search train {n_train:,} rows x {n_features} features, dev {n_dev:,} rows")

    rows = []
    for i, (kernel, c, gamma) in enumerate(combinations, start=1):
        print(f"[{i}/{total}] kernel={kernel:<7} C={c:<6} gamma={str(gamma):<6}")

        try:
            fit_seconds, n_support, scores = _fit_and_score_svm(
                kernel, c, gamma,
                search_train_x, search_train_y, search_dev_x, search_dev_y,
            )
        except Exception as error:
            # cuML's SVM solver rejects some combinations outright: a high C on a hard
            # pairwise sub-problem raises "Working set has already been initialized!"
            # from its C++ layer with the card nearly empty, so this is a solver bug
            # rather than exhaustion. There is no CPU retry - libsvm needs well over
            # 15 minutes for the combinations cuML refuses, against ~9s for the ones
            # it accepts, which would cost more than the whole rest of the grid. The
            # combination is recorded with NaN scores and the reason instead, so the
            # gap is visible in the results table and the remaining combinations -
            # which may well include the best one - still get scored.
            reason = f"{type(error).__name__}: {str(error).splitlines()[0][:120]}"
            print(f"  !! FAILED on GPU: {reason}")
            print("     skipped; continuing with the remaining combinations")
            free_gpu_memory()
            rows.append(
                {
                    "kernel": kernel,
                    "C": c,
                    "gamma": gamma,
                    "fit_seconds": float("nan"),
                    "n_support": float("nan"),
                    "error": reason,
                }
            )
            continue

        rows.append(
            {
                "kernel": kernel,
                "C": c,
                "gamma": gamma,
                "fit_seconds": fit_seconds,
                "n_support": n_support,
                **scores,
            }
        )
        print(
            f"  f1_macro={scores['f1_macro']:.4f}"
            f" accuracy={scores['accuracy']:.4f}"
            f" ({i}/{total} combinations done)"
        )

    results = pd.DataFrame(rows)
    failed = int(results["error"].notna().sum()) if "error" in results else 0
    print(f"Search finished: {total - failed}/{total} combinations scored", end="")
    print(f", {failed} failed" if failed else "")

    if "f1_macro" not in results:
        # Every combination failed, so there is nothing to rank - hand back the
        # reasons rather than dying on a missing sort key.
        return results
    # na_position defaults to "last", so failed rows sink below every scored one.
    return results.sort_values("f1_macro", ascending=False, ignore_index=True)

In [57]:
svm_search_results = search_svm(train_x, train_y, dev_x, dev_y)
dump_search_results(svm_search_results, "svm-search.csv")
display(svm_search_results)

SVM hyperparameter search over 9 combinations
Preparing search subsamples...
Subsampled 11,365,674 -> 49,992 rows across 15 classes
Subsampled 3,246,589 -> 100,497 rows across 15 classes
  search train 49,992 rows x 25 features, dev 100,497 rows
[1/9] kernel=rbf     C=0.1    gamma=scale 
  fitting on 49,992 rows ... done
  predicting 100,497 dev rows ... done
  f1_macro=0.5749 accuracy=0.9362 (1/9 combinations done)
[2/9] kernel=rbf     C=0.1    gamma=0.1   
  fitting on 49,992 rows ... done
  predicting 100,497 dev rows ... done
  f1_macro=0.6846 accuracy=0.9652 (2/9 combinations done)
[3/9] kernel=rbf     C=1.0    gamma=scale 
  fitting on 49,992 rows ... done
  predicting 100,497 dev rows ... done
  f1_macro=0.6877 accuracy=0.9641 (3/9 combinations done)
[4/9] kernel=rbf     C=1.0    gamma=0.1   
  fitting on 49,992 rows ... done
  predicting 100,497 dev rows ... done
  f1_macro=0.7615 accuracy=0.9776 (4/9 combinations done)
[5/9] kernel=rbf     C=10.0   gamma=scale 
  fitting on 49

,kernel,C,gamma,fit_seconds,n_support,accuracy,precision_macro,recall_macro,f1_macro,error
0,rbf,10.0,0.1,10.900169,12401.0,0.978815,0.836687,0.754204,0.770191,NaN
1,rbf,1.0,0.1,8.318841,17260.0,0.977631,0.861189,0.724085,0.761518,NaN
2,linear,1.0,scale,36.027224,11986.0,0.962039,0.872350,0.659094,0.699255,NaN
3,rbf,1.0,scale,8.030320,20458.0,0.964138,0.907165,0.631474,0.687740,NaN
4,rbf,0.1,0.1,10.494591,30330.0,0.965153,0.861869,0.621463,0.684619,NaN
5,linear,0.1,scale,10.144107,17655.0,0.956138,0.874304,0.603402,0.648513,NaN
6,rbf,0.1,scale,9.476785,40911.0,0.936177,0.741910,0.523855,0.574895,NaN
7,rbf,10.0,scale,NaN,NaN,NaN,NaN,NaN,NaN,RuntimeError: exception occurred! file=/tmp/co...
8,linear,10.0,scale,NaN,NaN,NaN,NaN,NaN,NaN,RuntimeError: exception occurred! file=/tmp/co...


In [58]:
best_svm = get_best_hyperparameter(svm_search_results)
SVM_MODEL_KERNEL = str(best_svm["kernel"])
SVM_MODEL_C = float(best_svm["C"])
SVM_MODEL_GAMMA = best_svm["gamma"]

Best by f1_macro:
            kernel = rbf
                 C = 10.0
             gamma = 0.1
       fit_seconds = 10.900169372558594
         n_support = 12401.0
          accuracy = 0.9788152880185478
   precision_macro = 0.8366866805375203
      recall_macro = 0.7542036077816197
          f1_macro = 0.7701906746235035
             error = nan


## Train SVM

In [59]:
def train_svm(
    train_x,
    train_y,
    kernel: str,
    c: float,
    gamma,
    train_samples: int | None = SVM_TRAIN_SAMPLES,
    random_state: int = SVM_RANDOM_STATE,
    subfolder: str = "SVM",
) -> SVC:
    print(f"Training SVM: kernel={kernel} C={c} gamma={gamma}")

    print("Preparing training subsample...")
    features, labels = stratified_subsample(
        train_x, train_y, train_samples, random_state
    )
    n_rows, n_features = features.shape

    model = build_svm(kernel, c, gamma)
    with step(f"fitting on {n_rows:,} rows x {n_features} features"):
        model.fit(features, labels)
    print(f"  {_support_vector_count(model):,.0f} support vectors")

    with step("saving model"):
        file_path = dump_trained_model(model, get_svm_name(kernel, c, gamma), subfolder)
    print(f"Saved model to: {file_path}")
    return model

In [60]:
train_svm(train_x, train_y, SVM_MODEL_KERNEL, SVM_MODEL_C, SVM_MODEL_GAMMA)

Training SVM: kernel=rbf C=10.0 gamma=0.1
Preparing training subsample...
Subsampled 11,365,674 -> 299,994 rows across 15 classes
  fitting on 299,994 rows x 25 features ... done
  45,515 support vectors
  saving model ... done
Saved model to: trained-model/SVM/svm-k-rbf-c-10.0-g-0.1.pkl


SVC()

## Evaluate SVM

In [64]:
def load_svm_model(model_path: str, folder_path: str = PATH_FOLDER_MODEL) -> SVC:
    return joblib.load(os.path.join(folder_path, "SVM", model_path))

In [65]:
def svm_predict(
    model_path,
    x,
    chunk_size: int = PREDICT_BY_CHUNK,
    cache_dir: str = PATH_FOLDER_PREDICTION_RESULT,
    cache_dir_subfolder: str = "",
):
    cache_dir = os.path.join(cache_dir, "SVM")
    if cache_dir_subfolder:
        cache_dir = os.path.join(cache_dir, cache_dir_subfolder)
    os.makedirs(cache_dir, exist_ok=True)

    with step(f"loading model {model_path}"):
        model = load_svm_model(model_path)

    total = len(x)
    if total == 0:
        print("Nothing to predict (0 rows)")
        return np.empty(0)

    n_chunks = math.ceil(total / chunk_size)
    print(
        f"Predicting {total:,} rows in {n_chunks:,} chunks of {chunk_size:,}"
        f" ({_support_vector_count(model):,.0f} support vectors)"
    )
    print(f"  cache: {cache_dir}")

    paths = []
    computed = 0
    cached = 0

    for i, start in enumerate(range(0, total, chunk_size), start=1):
        path = os.path.join(cache_dir, f"svm_{i:05d}.npy")
        paths.append(path)

        if os.path.exists(path):
            cached += 1
        else:
            if isinstance(x, (pl.DataFrame, pl.Series)):
                chunk = x.slice(start, chunk_size)
            else:
                chunk = x[start : start + chunk_size]

            pred = to_numpy(model.predict(to_features(chunk)))

            tmp = path + ".tmp"
            with open(tmp, "wb") as f:
                np.save(f, pred)
            os.replace(tmp, path)
            computed += 1

        progress_bar(i, n_chunks, f"chunks ({computed:,} computed, {cached:,} cached)")

    del model
    free_gpu_memory()

    with step(f"loading {len(paths):,} prediction chunks"):
        result = np.concatenate([np.load(p) for p in paths])
    return result

In [66]:
svm_predict(
    get_svm_name(SVM_MODEL_KERNEL, SVM_MODEL_C, SVM_MODEL_GAMMA),
    dev_x,
    cache_dir_subfolder="dev",
)

  loading model svm-k-rbf-c-10.0-g-0.1.pkl ... done
Predicting 3,246,589 rows in 325 chunks of 10,000 (45,515 support vectors)
  cache: prediction-result/SVM/dev
  [##############################] 325/325 chunks (325 computed, 0 cached)                          
  loading 325 prediction chunks ... done


array([11, 11, 11, ...,  0,  0,  0], shape=(3246589,), dtype=int32)

In [69]:
svm_predict(
    get_svm_name(SVM_MODEL_KERNEL, SVM_MODEL_C, SVM_MODEL_GAMMA),
    test_x,
    cache_dir_subfolder="test",
)

  loading model svm-k-rbf-c-10.0-g-0.1.pkl ... done
Predicting 3,246,594 rows in 325 chunks of 10,000 (45,515 support vectors)
  cache: prediction-result/SVM/test
  [##############################] 325/325 chunks (325 computed, 0 cached)                          
  loading 325 prediction chunks ... done


array([11, 11, 11, ...,  0,  0,  0], shape=(3246594,), dtype=int32)

### Report

In [93]:
get_evaluation_results(
    os.path.join(PATH_FOLDER_PREDICTION_RESULT, "SVM", "dev"),
    "dev",
)

Found 168 dev files


,accuracy,precision_macro,recall_macro,f1_macro
0,0.979212,0.737305,0.787156,0.699035


In [94]:
get_evaluation_results(
    os.path.join(PATH_FOLDER_PREDICTION_RESULT, "SVM", "test"),
    "test",
)

Found 168 test files


,accuracy,precision_macro,recall_macro,f1_macro
0,0.979236,0.735212,0.786074,0.700878


In [95]:
svm_test_pred, svm_test_true = load_predictions_and_labels(
    os.path.join(PATH_FOLDER_PREDICTION_RESULT, "SVM", "test"), "test"
)

Found 168 test files


In [104]:
get_classification_report(pred=svm_test_pred, true=svm_test_true)

/home/joshuans/miniforge3/envs/rapids/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,class,precision,recall,f1,support
0,Benign,0.988321,0.997111,0.992697,2696950
1,Bot,0.970477,0.997572,0.983838,57238
2,Brute Force -Web,0.703704,0.467213,0.561576,122
3,Brute Force -XSS,0.003384,0.673913,0.006733,46
4,DDOS attack-HOIC,0.999665,1.000000,0.999832,137203
5,DDOS attack-LOIC-UDP,0.703476,0.994220,0.823952,346
6,DDoS attacks-LOIC-HTTP,0.998701,0.940523,0.968739,115237
7,DoS attacks-GoldenEye,0.991086,0.830382,0.903644,8301
8,DoS attacks-Hulk,0.984528,0.999480,0.991948,92382
9,DoS attacks-SlowHTTPTest,0.759781,0.426192,0.546071,27978


In [105]:
cm = get_confusion_matrix(pred=svm_test_pred, true=svm_test_true)
display(cm)

/home/joshuans/miniforge3/envs/rapids/lib/python3.12/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


,Benign,Bot,Brute Force -Web,Brute Force -XSS,DDOS attack-HOIC,DDOS attack-LOIC-UDP,DDoS attacks-LOIC-HTTP,DoS attacks-GoldenEye,DoS attacks-Hulk,DoS attacks-SlowHTTPTest,DoS attacks-Slowloris,FTP-BruteForce,Infilteration,SQL Injection,SSH-Bruteforce
Benign,2689158,1736,23,2707,43,1,139,27,67,2,641,21,1400,956,29
Bot,139,57099,0,0,0,0,0,0,0,0,0,0,0,0,0
Brute Force -Web,40,0,57,14,0,0,0,0,0,0,0,0,0,11,0
Brute Force -XSS,15,0,0,31,0,0,0,0,0,0,0,0,0,0,0
DDOS attack-HOIC,0,0,0,0,137203,0,0,0,0,0,0,0,0,0,0
DDOS attack-LOIC-UDP,2,0,0,0,0,344,0,0,0,0,0,0,0,0,0
DDoS attacks-LOIC-HTTP,366,0,0,6344,0,144,108383,0,0,0,0,0,0,0,0
DoS attacks-GoldenEye,51,0,0,0,0,0,0,6893,1355,0,2,0,0,0,0
DoS attacks-Hulk,14,0,0,0,0,0,0,34,92334,0,0,0,0,0,0
DoS attacks-SlowHTTPTest,0,0,0,0,0,0,0,0,0,11924,0,16054,0,0,0
